<a href="https://colab.research.google.com/github/AneelGaddam8999/Data-Driven-Supply-Chain-Optimization-and-Inventory-Analytics-Using-the-DataCo-Dataset/blob/main/notebooks/01_Data_Cleaning_and_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
folder_path = '/content/drive/MyDrive/SupplyChainProject/'

df_main = pd.read_csv(folder_path + 'DataCoSupplyChainDataset.csv', encoding='latin1')
df_desc = pd.read_csv(folder_path + 'DescriptionDataCoSupplyChain.csv', encoding='latin1')
df_logs = pd.read_csv(folder_path + 'tokenized_access_logs.csv', encoding='utf-8')

print("All datasets loaded successfully!")

display(df_main.head(2))

All datasets loaded successfully!


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class


In [ ]:
df_main.columns = [col.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for col in df_main.columns]

print("--- Missing Values BEFORE Cleaning ---")
missing = df_main.isnull().sum()
print(missing[missing > 0])

cols_to_drop = ['product_description', 'product_image']
existing_cols = [c for c in cols_to_drop if c in df_main.columns]
df_clean = df_main.drop(columns=existing_cols)

if 'order_zipcode' in df_clean.columns:
    df_clean['order_zipcode'] = df_clean['order_zipcode'].fillna(0)
if 'customer_zipcode' in df_clean.columns:
    df_clean['customer_zipcode'] = df_clean['customer_zipcode'].fillna(0)

date_columns = ['shipping_date_dateorders', 'order_date_dateorders']
for col in date_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

print("\n--- Missing Values AFTER Cleaning ---")
missing_after = df_clean.isnull().sum()
print(missing_after[missing_after > 0])
print("\nData cleaning complete! Shape of clean dataset:", df_clean.shape)

--- Missing Values BEFORE Cleaning ---
customer_lname              8
customer_zipcode            3
order_zipcode          155679
product_description    180519
dtype: int64

--- Missing Values AFTER Cleaning ---
customer_lname    8
dtype: int64

Data cleaning complete! Shape of clean dataset: (180519, 51)


In [ ]:
import numpy as np

df_clean['customer_lname'] = df_clean['customer_lname'].fillna('Unknown')
df_clean['is_late'] = (df_clean['delivery_status'] == 'Late delivery').astype(int)
df_clean['profit_margin_pct'] = np.where(
    df_clean['sales_per_customer'] > 0,
    (df_clean['benefit_per_order'] / df_clean['sales_per_customer']) * 100,
    0
)

export_path = '/content/drive/MyDrive/SupplyChainProject/Cleaned_DataCo.csv'
df_clean.to_csv(export_path, index=False)

print("Remaining missing values:", df_clean.isnull().sum().sum())
print("New features added: 'is_late' and 'profit_margin_pct'")
print(f"Clean dataset ready for SQL and Power BI! Saved to: {export_path}")

Remaining missing values: 0
New features added: 'is_late' and 'profit_margin_pct'
Clean dataset ready for SQL and Power BI! Saved to: /content/drive/MyDrive/SupplyChainProject/Cleaned_DataCo.csv
